# Data Exploration

---
## 5. Audio Preprocessing Pipeline

Demonstrates the preprocessing steps defined in `asr_config.yaml`:
normalization, silence removal, and the effect on signal quality.

In [ ]:
def normalize_audio(signal: np.ndarray) -> np.ndarray:
    """Peak normalization to [-1, 1] (matches ASRProcessor._process_chunk)."""
    max_val = np.max(np.abs(signal))
    if max_val > 0:
        return signal / max_val
    return signal


def remove_silence(signal: np.ndarray, sr: int, threshold_db: float = -40,
                   min_silence_dur: float = 0.3) -> np.ndarray:
    """
    Remove silence segments based on energy threshold.
    Uses config values: silence_threshold=-40dB, min_silence_duration=0.3s
    """
    frame_len = int(0.025 * sr)
    hop_len = int(0.010 * sr)
    min_silence_frames = int(min_silence_dur * sr / hop_len)

    # Calculate frame energies in dB
    n_frames = 1 + (len(signal) - frame_len) // hop_len
    energies_db = np.array([
        20 * np.log10(np.sqrt(np.mean(signal[i*hop_len:i*hop_len+frame_len]**2)) + 1e-10)
        for i in range(n_frames)
    ])

    # Identify voiced frames
    voiced_mask = energies_db > threshold_db

    # Reconstruct signal from voiced frames only
    voiced_samples = []
    for i, is_voiced in enumerate(voiced_mask):
        if is_voiced:
            start = i * hop_len
            end = min(start + frame_len, len(signal))
            voiced_samples.append(signal[start:end])

    if voiced_samples:
        return np.concatenate(voiced_samples)
    return signal  # fallback: return original if everything is "silent"


# Apply preprocessing
silence_thresh = audio_cfg.get('silence_threshold', -40)
min_sil_dur = audio_cfg.get('min_silence_duration', 0.3)

audio_normalized = normalize_audio(audio_signal)
audio_trimmed = remove_silence(audio_normalized, SR, threshold_db=silence_thresh, min_silence_dur=min_sil_dur)

print(f"\n🔧 Preprocessing Results:")
print(f"   Original:    {len(audio_signal):>8,} samples ({len(audio_signal)/SR:.2f}s)")
print(f"   Normalized:  {len(audio_normalized):>8,} samples (peak={np.max(np.abs(audio_normalized)):.4f})")
print(f"   Trimmed:     {len(audio_trimmed):>8,} samples ({len(audio_trimmed)/SR:.2f}s)")
print(f"   Removed:     {(1 - len(audio_trimmed)/len(audio_signal))*100:.1f}% silence")

In [ ]:
# Visualize before/after preprocessing
fig, axes = plt.subplots(2, 1, figsize=(14, 6))

t_orig = np.arange(len(audio_signal)) / SR
t_trim = np.arange(len(audio_trimmed)) / SR

axes[0].plot(t_orig, audio_signal, color=COLORS['secondary'], linewidth=0.5, alpha=0.7)
axes[0].set_title('Before Preprocessing (Raw)', fontweight='bold')
axes[0].set_ylabel('Amplitude')
axes[0].set_xlabel('Time (s)')
axes[0].axhline(y=0, color='white', linewidth=0.3, alpha=0.3)

axes[1].plot(t_trim, audio_trimmed, color=COLORS['accent'], linewidth=0.5, alpha=0.7)
axes[1].set_title(f'After Preprocessing (Normalized + Silence Removed @ {silence_thresh} dB)', fontweight='bold')
axes[1].set_ylabel('Amplitude')
axes[1].set_xlabel('Time (s)')
axes[1].axhline(y=0, color='white', linewidth=0.3, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 6. Feature Extraction Deep Dive

Whisper uses **80-channel log-Mel spectrogram** as input features.
Let's visualize the Mel filterbank and compare raw vs. log-scaled features.

In [ ]:
if HAS_LIBROSA:
    fig, axes = plt.subplots(2, 2, figsize=(14, 9))

    # --- 6a. Mel Filterbank ---
    mel_fb = librosa.filters.mel(sr=SR, n_fft=n_fft, n_mels=n_mels,
                                  fmin=audio_cfg.get('fmin', 50),
                                  fmax=audio_cfg.get('fmax', 7600))
    for i in range(0, n_mels, max(1, n_mels // 10)):
        axes[0, 0].plot(mel_fb[i], alpha=0.7, linewidth=0.8)
    axes[0, 0].set_title(f'Mel Filterbank ({n_mels} filters)', fontweight='bold')
    axes[0, 0].set_xlabel('FFT Bin')
    axes[0, 0].set_ylabel('Weight')

    # --- 6b. Log-Mel vs Linear Mel (single frame comparison) ---
    mid_frame = mel_spec.shape[1] // 2
    mel_linear = mel_spec[:, mid_frame]
    mel_log = mel_db[:, mid_frame]

    axes[0, 1].bar(range(n_mels), mel_linear, color=COLORS['info'], alpha=0.7, width=0.8)
    axes[0, 1].set_title('Single Frame — Linear Mel Energy', fontweight='bold')
    axes[0, 1].set_xlabel('Mel Band')
    axes[0, 1].set_ylabel('Energy')

    axes[1, 0].bar(range(n_mels), mel_log, color=COLORS['warning'], alpha=0.7, width=0.8)
    axes[1, 0].set_title('Single Frame — Log-Mel Energy (Whisper Input)', fontweight='bold')
    axes[1, 0].set_xlabel('Mel Band')
    axes[1, 0].set_ylabel('dB')

    # --- 6c. MFCC Statistics ---
    mfcc_mean = np.mean(mfccs, axis=1)
    mfcc_std = np.std(mfccs, axis=1)
    x_pos = np.arange(n_mfcc)
    axes[1, 1].bar(x_pos, mfcc_mean, yerr=mfcc_std, color=COLORS['primary'],
                   alpha=0.7, capsize=3, ecolor=COLORS['secondary'])
    axes[1, 1].set_title('MFCC Distribution (mean ± std)', fontweight='bold')
    axes[1, 1].set_xlabel('MFCC Coefficient')
    axes[1, 1].set_ylabel('Value')
    axes[1, 1].set_xticks(x_pos)

    plt.tight_layout()
    plt.show()
else:
    print('⚠️  Install librosa for feature extraction visualizations.')

---
## 7. ASR Processor Module Exploration

Test the project's own `ASRProcessor` class from `src/voice_assistant/asr/processor.py`.

In [ ]:
try:
    from voice_assistant.asr.processor import ASRProcessor

    # Initialize with project config values
    processor = ASRProcessor(
        sample_rate=SR,
        chunk_size=audio_cfg.get('hop_length', 160),
        timeout=5
    )

    # Process a chunk through the project's pipeline
    chunk = audio_signal[:SR]  # 1 second of audio
    processor.add_audio_chunk(chunk)

    # Extract features using project's own method
    features = processor._process_chunk(chunk)

    print(f"\n🔬 ASRProcessor Results:")
    print(f"   Input chunk:   {chunk.shape} ({len(chunk)/SR:.2f}s)")
    print(f"   Output shape:  {features.shape}")
    print(f"   Feature range: [{features.min():.4f}, {features.max():.4f}]")
    print(f"   Feature mean:  {features.mean():.4f}")
    print(f"   Feature std:   {features.std():.4f}")

    # Plot the FFT features extracted by ASRProcessor
    fig, ax = plt.subplots(figsize=(14, 4))
    freqs = np.linspace(0, SR / 2, len(features))
    ax.fill_between(freqs, features, color=COLORS['primary'], alpha=0.4)
    ax.plot(freqs, features, color=COLORS['primary'], linewidth=0.8)
    ax.set_title('ASRProcessor._extract_features() — Magnitude Spectrum', fontweight='bold')
    ax.set_xlabel('Frequency (Hz)')
    ax.set_ylabel('Magnitude')
    ax.set_xlim(0, SR / 2)
    plt.tight_layout()
    plt.show()

    processor.reset()

except ImportError as e:
    print(f"⚠️  Could not import ASRProcessor: {e}")
    print("   Make sure 'src/' is in sys.path and dependencies are installed.")

---
## 8. Dataset Quality Checks

Functions to audit audio data quality — useful when building or curating
datasets for ASR training or evaluation.

In [ ]:
def compute_snr_estimate(signal: np.ndarray, sr: int, noise_floor_percentile: int = 10) -> float:
    """
    Estimate Signal-to-Noise Ratio (SNR) in dB.
    Uses the quietest frames as a noise floor estimate.
    """
    frame_len = int(0.025 * sr)
    hop_len = int(0.010 * sr)
    n_frames = 1 + (len(signal) - frame_len) // hop_len

    frame_energies = np.array([
        np.mean(signal[i*hop_len:i*hop_len+frame_len]**2)
        for i in range(n_frames)
    ])

    noise_energy = np.percentile(frame_energies, noise_floor_percentile)
    signal_energy = np.mean(frame_energies)

    if noise_energy > 0:
        return 10 * np.log10(signal_energy / noise_energy)
    return float('inf')


def detect_clipping(signal: np.ndarray, threshold: float = 0.99) -> dict:
    """Detect potential audio clipping."""
    clipped_samples = np.sum(np.abs(signal) >= threshold)
    clipped_pct = (clipped_samples / len(signal)) * 100
    return {
        'clipped_samples': int(clipped_samples),
        'clipped_pct': clipped_pct,
        'is_clipped': clipped_pct > 0.1,  # >0.1% is concerning
    }


def compute_dc_offset(signal: np.ndarray) -> float:
    """Compute DC offset (mean of signal)."""
    return float(np.mean(signal))


# Run quality checks on our audio
snr = compute_snr_estimate(audio_signal, SR)
clipping = detect_clipping(audio_signal)
dc_offset = compute_dc_offset(audio_signal)

print(f"\n🏥 Audio Quality Report:")
print(f"   SNR (estimated):    {snr:.1f} dB")
print(f"   DC Offset:          {dc_offset:.6f}")
print(f"   Clipped Samples:    {clipping['clipped_samples']:,} ({clipping['clipped_pct']:.3f}%)")
print(f"   Clipping Detected:  {'⚠️ YES' if clipping['is_clipped'] else '✅ NO'}")
print(f"   Peak Level:         {20 * np.log10(np.max(np.abs(audio_signal)) + 1e-10):.1f} dBFS")
print(f"   RMS Level:          {20 * np.log10(np.sqrt(np.mean(audio_signal**2)) + 1e-10):.1f} dBFS")

---
## 9. Simulated Batch Dataset Statistics

Simulate analyzing a batch of audio files (varying durations, noise levels)
to demonstrate the kind of EDA you'd run on a real ASR dataset.

In [ ]:
np.random.seed(42)
N_SAMPLES = 200

# Simulate dataset metadata
dataset = {
    'duration_s': np.random.lognormal(mean=1.5, sigma=0.6, size=N_SAMPLES).clip(0.5, 30),
    'snr_db': np.random.normal(loc=20, scale=8, size=N_SAMPLES).clip(0, 50),
    'rms_dbfs': np.random.normal(loc=-18, scale=5, size=N_SAMPLES).clip(-40, 0),
    'clipped_pct': np.random.exponential(scale=0.05, size=N_SAMPLES).clip(0, 5),
    'sample_rate': np.random.choice([8000, 16000, 22050, 44100], size=N_SAMPLES, p=[0.05, 0.70, 0.15, 0.10]),
}

fig, axes = plt.subplots(2, 3, figsize=(16, 9))

# Duration Distribution
axes[0, 0].hist(dataset['duration_s'], bins=30, color=COLORS['primary'], alpha=0.75, edgecolor='white', linewidth=0.5)
axes[0, 0].axvline(np.median(dataset['duration_s']), color=COLORS['secondary'], linestyle='--', linewidth=2, label=f"median={np.median(dataset['duration_s']):.1f}s")
axes[0, 0].set_title('Duration Distribution', fontweight='bold')
axes[0, 0].set_xlabel('Duration (s)')
axes[0, 0].set_ylabel('Count')
axes[0, 0].legend()

# SNR Distribution
axes[0, 1].hist(dataset['snr_db'], bins=30, color=COLORS['accent'], alpha=0.75, edgecolor='white', linewidth=0.5)
axes[0, 1].axvline(10, color=COLORS['secondary'], linestyle='--', linewidth=2, label='Min recommended (10 dB)')
axes[0, 1].set_title('SNR Distribution', fontweight='bold')
axes[0, 1].set_xlabel('SNR (dB)')
axes[0, 1].set_ylabel('Count')
axes[0, 1].legend()

# RMS Level
axes[0, 2].hist(dataset['rms_dbfs'], bins=30, color=COLORS['warning'], alpha=0.75, edgecolor='white', linewidth=0.5)
axes[0, 2].set_title('RMS Level Distribution', fontweight='bold')
axes[0, 2].set_xlabel('RMS (dBFS)')
axes[0, 2].set_ylabel('Count')

# Sample Rate Breakdown
sr_values, sr_counts = np.unique(dataset['sample_rate'], return_counts=True)
bars = axes[1, 0].bar([f"{sr//1000}k" for sr in sr_values], sr_counts,
                       color=[COLORS['primary'], COLORS['accent'], COLORS['warning'], COLORS['secondary']][:len(sr_values)],
                       alpha=0.8, edgecolor='white')
axes[1, 0].set_title('Sample Rate Breakdown', fontweight='bold')
axes[1, 0].set_xlabel('Sample Rate')
axes[1, 0].set_ylabel('Count')
for bar, count in zip(bars, sr_counts):
    axes[1, 0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                    str(count), ha='center', fontweight='bold')

# Clipping Analysis
axes[1, 1].hist(dataset['clipped_pct'], bins=30, color=COLORS['secondary'], alpha=0.75, edgecolor='white', linewidth=0.5)
axes[1, 1].axvline(0.1, color=COLORS['warning'], linestyle='--', linewidth=2, label='Clipping threshold (0.1%)')
axes[1, 1].set_title('Clipping Percentage', fontweight='bold')
axes[1, 1].set_xlabel('Clipped Samples (%)')
axes[1, 1].set_ylabel('Count')
axes[1, 1].legend()

# Duration vs SNR scatter
scatter = axes[1, 2].scatter(dataset['duration_s'], dataset['snr_db'],
                              c=dataset['rms_dbfs'], cmap='viridis', alpha=0.6,
                              s=20, edgecolors='white', linewidth=0.3)
axes[1, 2].set_title('Duration vs SNR (colored by RMS)', fontweight='bold')
axes[1, 2].set_xlabel('Duration (s)')
axes[1, 2].set_ylabel('SNR (dB)')
fig.colorbar(scatter, ax=axes[1, 2], label='RMS (dBFS)', pad=0.01)

plt.tight_layout()
plt.show()

# Summary statistics
print(f"\n📊 Dataset Summary ({N_SAMPLES} samples):")
print(f"   Total duration:   {np.sum(dataset['duration_s']):.0f}s ({np.sum(dataset['duration_s'])/60:.1f} min)")
print(f"   Mean duration:    {np.mean(dataset['duration_s']):.1f}s")
print(f"   Mean SNR:         {np.mean(dataset['snr_db']):.1f} dB")
print(f"   Low-SNR samples:  {np.sum(dataset['snr_db'] < 10)} ({np.sum(dataset['snr_db'] < 10)/N_SAMPLES*100:.1f}%)")
print(f"   Clipped samples:  {np.sum(dataset['clipped_pct'] > 0.1)} ({np.sum(dataset['clipped_pct'] > 0.1)/N_SAMPLES*100:.1f}%)")
print(f"   Non-16kHz:        {np.sum(dataset['sample_rate'] != 16000)} (need resampling)")

---
## 10. Whisper Model Architecture Overview

Summary of the Whisper model architecture used in this project,
including input/output specifications and parameter counts.

In [ ]:
# Whisper model specifications (from OpenAI)
whisper_models = {
    'Model': ['tiny', 'base', 'small', 'medium', 'large-v3', 'large-v3-turbo'],
    'Parameters': ['39M', '74M', '244M', '769M', '1550M', '809M'],
    'English-only WER': ['~7.6%', '~5.0%', '~3.4%', '~2.7%', '~2.0%', '~2.1%'],
    'Relative Speed': ['32x', '16x', '6x', '2x', '1x', '~4x'],
    'VRAM (FP16)': ['~1 GB', '~1 GB', '~2 GB', '~5 GB', '~10 GB', '~6 GB'],
    'Layers': [4, 6, 12, 24, 32, 4],
    'Width': [384, 512, 768, 1024, 1280, 1280],
    'Heads': [6, 8, 12, 16, 20, 20],
}

# Highlight the model used in this project
project_model = 'large-v3-turbo'  # from main.py
config_model = model_cfg.get('name', 'openai/whisper-base').split('/')[-1] if model_cfg else 'whisper-base'

print('=' * 80)
print('OpenAI Whisper Model Family')
print('=' * 80)
print(f"{'Model':<18} {'Params':<10} {'WER':<14} {'Speed':<10} {'VRAM':<10} {'Layers':<8} {'Width':<8} {'Heads':<6}")
print('-' * 80)
for i in range(len(whisper_models['Model'])):
    name = whisper_models['Model'][i]
    marker = ' ◀ PROJECT' if name == project_model else ''
    marker2 = ' ◀ CONFIG' if name.replace('whisper-', '') == config_model.replace('whisper-', '') else ''
    print(f"  {name:<16} {whisper_models['Parameters'][i]:<10} {whisper_models['English-only WER'][i]:<14} "
          f"{whisper_models['Relative Speed'][i]:<10} {whisper_models['VRAM (FP16)'][i]:<10} "
          f"{whisper_models['Layers'][i]:<8} {whisper_models['Width'][i]:<8} {whisper_models['Heads'][i]:<6}{marker}{marker2}")

print(f"\n🎯 main.py uses:          {project_model}")
print(f"🎯 asr_config.yaml uses:  {config_model}")
if project_model.replace('whisper-', '') != config_model.replace('whisper-', ''):
    print(f"⚠️  Note: main.py and asr_config.yaml reference different models!")